In [1]:
if (!requireNamespace("mixOmics", quietly = TRUE)) {
    options(BioC_mirror = "https://bioconductor.org")
    BiocManager::install(version = "3.18", force = TRUE)
    BiocManager::install("mixOmics")
}

library(mixOmics)
library(BiocParallel)

Loading required package: MASS

Loading required package: lattice

Loading required package: ggplot2


Loaded mixOmics 6.26.0
Thank you for using mixOmics!
Tutorials: http://mixomics.org
Bookdown vignette: https://mixomicsteam.github.io/Bookdown
Questions, issues: Follow the prompts at http://mixomics.org/contact-us
Cite us:  citation('mixOmics')




In [2]:
DATA_DIR <- "/home/mariya/Data/pallas/mlomix_datasets"

TRAIN_DATA_DIR <- file.path(DATA_DIR, "train-bimodal")
METH_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.dnam.csv")
GEX_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.gex.csv")
LABELS_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "labels.csv")

TEST_DATA_DIR <- file.path(DATA_DIR, "test-bimodal")
METH_TEST_FILE <- file.path(TEST_DATA_DIR, "features.dnam.csv")
GEX_TEST_FILE <- file.path(TEST_DATA_DIR, "features.gex.csv")
LABELS_TEST_FILE <- file.path(TEST_DATA_DIR, "labels.csv")

OUTPUT_DIR <- file.path("/home/mariya/Data/pallas", "mixOmics")

In [3]:
source("helpers.R")
train_data <- data(GEX_TRAIN_FILE, METH_TRAIN_FILE, LABELS_TRAIN_FILE)

Samples: 827 
GEX features: 19499 
DNAm features: 186411 
Subtypes: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 AML, CBFB::MYH11 AML, CEBPAalt AML, DEK::NUP214 AML, ETV6-r AML, KAT6A-r AML, KMT2A-r AML, NPM1alt AML, NUP98-r AML, RUNX1::RUNX1T1 B-ALL, BCR::ABL1 B-ALL, BCR::ABL1-like B-ALL, DUX4-r B-ALL, ETV6::RUNX1 B-ALL, ETV6::RUNX1-like B-ALL, high hyperdiploid B-ALL, iAMP21 B-ALL, KMT2A-r B-ALL, MEF2D-r B-ALL, PAX5alt B-ALL, TCF3::PBX1 B-ALL, ZNF384-r Non-leukemic T-ALL 


In [4]:
X_gex_train <- train_data$X_gex
X_dnam_train <- train_data$X_dnam
Y_train     <- train_data$Y

head(X_gex_train)
head(X_dnam_train)
head(Y_train)

,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167,⋯,ENSG00000288825,ENSG00000288859,ENSG00000288867,ENSG00000289549,ENSG00000289604,ENSG00000290292,ENSG00000291237,ENSG00000293137,ENSG00000293560,ENSG00000310560
ALL_1005,-1.1224793,-1.122479,4.109307,4.278200,4.194208,4.861957,-1.1224793,6.7128210,4.100312,4.528643,⋯,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.1224793,-1.1224793,5.897522
ALL_1019,-1.1224793,-1.122479,3.830802,4.183227,4.025231,8.290267,1.0262774,6.5421577,4.981583,4.307345,⋯,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.1224793,-1.1224793,7.336350
ALL_1039,-0.6669992,-1.122479,2.812976,3.102466,2.856878,7.699790,6.4384579,6.6288329,4.450187,3.735083,⋯,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.1224793,-1.1224793,6.942343
ALL_1047,-0.8916376,-1.122479,4.626464,2.600433,3.590264,2.981393,-1.0381515,2.5370304,3.685318,5.032487,⋯,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.1224793,-0.8347985,5.688093
ALL_1056,1.6121671,-1.122479,3.044414,3.942390,3.210562,3.404924,-1.1224793,3.0858164,4.654257,4.678591,⋯,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,1.9215712,-1.1224793,5.303055
ALL_1060,-0.8753711,-1.122479,4.453345,2.020495,2.454826,2.605110,0.6144054,0.6213076,4.210564,4.662769,⋯,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-1.122479,-0.1276339,-0.8149354,5.106765


,cg00000029,cg00000109,cg00000165,cg00000236,cg00000289,cg00000292,cg00000321,cg00000363,cg00000714,cg00000721,⋯,ch.4.194519R,ch.4.2662600F,ch.4.3427878R,ch.6.115952F,ch.6.1693624F,ch.6.92327598R,ch.7.130270068F,ch.7.1839793F,ch.9.2043201R,ch.9.31239617R
ALL_1005,0.8918,0.1351,0.5648,0.9069,0.7154,0.9421,0.6087,0.7048,0.1825,0.9242,⋯,0.0157,0.1454,0.0731,0.0645,0.0176,0.1499,0.0508,0.1041,0.0352,0.2130
ALL_1019,0.8956,0.2010,0.1756,0.8664,0.8093,0.5549,0.6161,0.6669,0.1560,0.6668,⋯,0.0169,0.0441,0.0443,0.0288,0.0526,0.0841,0.0240,0.0455,0.0234,0.0313
ALL_1039,0.8885,0.8979,0.5445,0.9214,0.8003,0.9291,0.5742,0.5542,0.3055,0.9337,⋯,0.0154,0.0856,0.0478,0.0340,0.0940,0.0857,0.0305,0.0572,0.0344,0.0467
ALL_1047,0.8680,0.7204,0.1650,0.8162,0.5892,0.9283,0.7841,0.1283,0.2686,0.8471,⋯,0.0130,0.1966,0.1105,0.0444,0.1768,0.1339,0.0387,0.1298,0.0901,0.1271
ALL_1056,0.6973,0.9095,0.1237,0.9202,0.6475,0.3749,0.8378,0.1015,0.0685,0.9423,⋯,0.0130,0.0914,0.0544,0.0346,0.0793,0.0777,0.0247,0.0770,0.0380,0.0481
ALL_1060,0.8838,0.9175,0.0870,0.9140,0.6780,0.9221,0.7761,0.9120,0.0981,0.9327,⋯,0.0167,0.0577,0.0703,0.0287,0.0613,0.0943,0.0306,0.0880,0.0297,0.0397


[1] B-ALL, MEF2D-r          B-ALL, ZNF384-r         B-ALL, BCR::ABL1       
[4] B-ALL, ETV6::RUNX1-like B-ALL, MEF2D-r          B-ALL, ETV6::RUNX1     
25 Levels: AML, APL-PML::RARA AML, CBFA2T3::GLIS2 ... T-ALL

In [5]:
NCOMP <- 30
KEEP_X <- 200

gex_model <- splsda(X_gex_train, Y_train,
                     ncomp = NCOMP,
                     keepX = rep(KEEP_X, NCOMP))

dnam_model <- splsda(X_dnam_train, Y_train,
                      ncomp = NCOMP,
                      keepX = rep(KEEP_X, NCOMP))

diablo_model <- block.splsda(list(gex = X_gex_train, dnam = X_dnam_train), Y_train,
                          ncomp = NCOMP,
                          keepX = list(gex = rep(KEEP_X, NCOMP),
                                       dnam = rep(KEEP_X, NCOMP)))

In [6]:
saveRDS(gex_model,    file.path(OUTPUT_DIR, "gex_model.rds"))
saveRDS(dnam_model,   file.path(OUTPUT_DIR, "dnam_model.rds"))
saveRDS(diablo_model, file.path(OUTPUT_DIR, "diablo_model.rds"))